# 05 · Modelo base: agente y evaluación en validación

**Qué contiene.** Carga el modelo base con Unsloth (4 bits) y evalúa el agente sobre validación con un prompt de sistema optimizado. Celdas 0–1: prueba inicial de carga y de generación. Celdas 2–3: primera versión de la corrida. Celdas 5–7: diagnóstico del límite de 2.048 tokens y recarga con 4.096. Celda 8: corrida vigente (versión 2). Celdas 4 y 9: revisión y análisis, sin GPU.

| | |
|---|---|
| **Runtime** | GPU T4 (Entorno de ejecución → Cambiar tipo de entorno → T4 GPU) |
| **Entradas (`MyDrive/qlora-series-ar/data/`)** | `catalogo_busqueda.csv`, `eval_val_final2.csv`, `specs_val_v2.csv`, `tools_v1.json` |
| **Salidas** | `baseline_val_tecnica_completa_optimizado.csv` y `_episodios.jsonl`; archivos `_parcial` durante la corrida |
| **Secretos de Colab** | `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY` |
| **Costo y tiempo** | Sin costo de API. La validación completa tomó unos 50 minutos en T4 |
| **Orden** | Después de `04`; antes de `06` |
| **En el repositorio** | `agente.py`, `corrida.py`, `modelo.py`, `scripts/evaluar.py` |

**Advertencias.**
- Las celdas 2–3 se conservan como historia: la corrida vigente es la celda 8. Correr solo una de las dos.
- La celda 0 carga la ruta `qlora_completo/adapter` (el adaptador) con 2.048 tokens: es una prueba de carga previa; la carga del modelo base para evaluar es la celda 2.


In [ ]:
!pip -q install unsloth
print("CELDA: MODELO BASE CON UNSLOTH - PRIMERA GENERACIÓN CON HERRAMIENTAS")
import json, re, time
import pandas as pd
import torch
from google.colab import drive
from unsloth import FastLanguageModel

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO HAY GPU: cambiá el runtime a T4")

MODELO = f"{RUTA}/qlora_completo/adapter"
t0 = time.time()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODELO, max_seq_length=2048, load_in_4bit=True)
FastLanguageModel.for_inference(model)
print(f"Cargado en {time.time() - t0:.0f}s | memoria de GPU: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

tools = json.load(open(f"{RUTA}/tools_v1.json"))
V = pd.read_csv(f"{RUTA}/eval_val_final2.csv")
V = V[V["marca_var"].fillna("") != "var_sobre_tasa"]
muestra = pd.concat([V[V["tipo"] == t].sample(1, random_state=4)
                     for t in ["ultimo_dato", "valor_puntual", "variacion_interanual", "sin_datos"]])

def generar(mensajes, max_new_tokens=300):
    ids = tokenizer.apply_chat_template(mensajes, tools=tools, add_generation_prompt=True,
                                        return_tensors="pt").to("cuda")
    t = time.time()
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False)
    nuevos = out[0][ids.shape[1]:]
    return tokenizer.decode(nuevos, skip_special_tokens=False), len(nuevos), time.time() - t, ids.shape[1]

for r in muestra.to_dict("records"):
    txt, n, seg, n_in = generar([{"role": "user", "content": r["tecnica"]}])
    print("\n" + "=" * 70)
    print(f"[{r['tipo']}] {r['tecnica']}")
    print(f"tokens de entrada: {n_in} | generados: {n} | {seg:.1f}s | {n / seg:.0f} tok/s")
    print("MODELO:", txt.strip()[:600])
    m = re.search(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", txt, re.S)
    try:
        llamada = json.loads(m.group(1)) if m else None
    except Exception:
        llamada = "JSON inválido"
    print("Llamada detectada:", llamada)

In [ ]:
print("Tiene adaptador:", hasattr(model, "peft_config"), "| max_seq_length:", model.max_seq_length)

In [ ]:
!pip -q install unsloth langfuse
print("CELDA 1 de 2: PREPARACIÓN (MODELO, LANGFUSE, ARCHIVOS)")
from unsloth import FastLanguageModel
import os, gc, json, shutil
import torch
import logging, warnings
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
from google.colab import drive, userdata

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
MODELO = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"
MAX_SEQ = 4096

if globals().get("MAX_SEQ_CARGADO") != MAX_SEQ:
    if "model" in globals():
        del model
        gc.collect()
        torch.cuda.empty_cache()
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODELO, max_seq_length=MAX_SEQ, load_in_4bit=True)
    FastLanguageModel.for_inference(model)
    MAX_SEQ_CARGADO = MAX_SEQ
print(f"Modelo listo | contexto {MAX_SEQ_CARGADO} | memoria de GPU: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

# ---------- Langfuse (cuenta de EE.UU.) ----------
HOST = "https://us.cloud.langfuse.com"
os.environ["LANGFUSE_PUBLIC_KEY"] = userdata.get("LANGFUSE_PUBLIC_KEY").strip()
os.environ["LANGFUSE_SECRET_KEY"] = userdata.get("LANGFUSE_SECRET_KEY").strip()
os.environ["LANGFUSE_HOST"] = HOST
os.environ["LANGFUSE_BASE_URL"] = HOST
from langfuse import get_client
lf = get_client()
print("Langfuse auth_check:", lf.auth_check())

# ---------- Archivos de trabajo (copias locales) ----------
for nombre in ["cache_api.db", "busqueda.db"]:
    if not os.path.exists(f"/content/{nombre}"):
        shutil.copy(f"{RUTA}/{nombre}", f"/content/{nombre}")
tools = json.load(open(f"{RUTA}/tools_v1.json"))
print("Herramientas:", [t["function"]["name"] for t in tools])

In [ ]:
print("CELDA 2 de 2: AGENTE + EVALUADOR + CORRIDA CON LANGFUSE")
assert all(x in globals() for x in ["model", "tokenizer", "lf", "RUTA", "tools"]), "Ejecutá antes la CELDA 1 de 2"
import re, json, time, sqlite3, shutil, unicodedata
from collections import Counter
import numpy as np
import pandas as pd
import requests
import torch

# ---------- Configuración ----------
SPLIT = "val"
ESTILO = "tecnica"            # "tecnica" o "coloquial"
N_PRUEBA = None               # None = todo el split
PROMPTS = ["optimizado"]      # luego, en otra corrida: ["basico"] con N_MUESTRA_BASICO
N_MUESTRA_BASICO = 100        # solo se usa cuando PROMPTS == ["basico"]
MAX_PASOS = 6
MAX_NUEVOS = 300
SAVE_CACHE = True
ETIQUETA_MODELO = "qwen3-4b-instruct-2507-base"
BASE = "https://apis.datos.gob.ar/series/api/series"
HEADERS = {"User-Agent": "Mozilla/5.0"}

PROMPT_OPT = """Sos un asistente que responde preguntas sobre estadísticas oficiales de Argentina usando tres herramientas. Seguí siempre este procedimiento:
1. Llamá a search_series con 2 a 6 palabras clave del tema (sin fechas ni números). Elegí, entre los candidatos devueltos, la serie cuyo título, distribución y frecuencia (mensual, trimestral o anual) mejor coincidan con lo que se pide. Usá solo ids devueltos por search_series; nunca inventes ids.
2. Llamá a get_series con la serie elegida:
- Valor de un período: start_date y end_date iguales, en formato AAAA (series anuales) o AAAA-MM.
- Último dato disponible: limit=1 y sort="desc", sin fechas.
- Variación interanual: representation_mode="percent_change_a_year_ago", start_date un año antes del período pedido y end_date el período pedido, con fechas completas AAAA-MM-DD (día 01; en series trimestrales, el primer mes del trimestre). La API devuelve una fracción (0.05 equivale a 5%) y la última fila corresponde al período pedido.
3. Llamá a submit_answer con la respuesta, el valor y la fecha del dato (formato AAAA-MM, o AAAA si la serie es anual). Para variaciones, el valor va en porcentaje (la fracción multiplicada por 100).
Si get_series devuelve data vacía, la API no tiene datos de esa serie para ese período: entregá submit_answer con valor y fecha en null y explicalo en la respuesta. No inventes valores."""
SISTEMA = {"basico": None, "optimizado": PROMPT_OPT}

def norm(s):
    s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s.lower().replace("_", " ")).strip()

STOP = {"de", "del", "la", "el", "los", "las", "en", "y", "a", "por", "que", "cual",
        "fue", "cuanto", "es", "un", "una", "para", "con", "se", "al", "lo"}
PARAMS_TOOL = {"ids", "start_date", "end_date", "representation_mode", "limit", "sort"}
cat = (pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
         .drop_duplicates("serie_id").set_index("serie_id"))

def clave(params):
    p = {k: str(v) for k, v in params.items() if v is not None}
    return json.dumps(dict(sorted(p.items())), ensure_ascii=False)

# ---------- Herramientas: caché primero, API real si falta ----------
class Herramientas:
    def __init__(self):
        self.cache = sqlite3.connect("/content/cache_api.db")
        self.fts = sqlite3.connect("/content/busqueda.db")
        self.fallos_cache, self.sin_red = 0, 0

    def search_series(self, query, k=10):
        toks = [t for t in re.findall(r"[a-z0-9]+", norm(query)) if t not in STOP and len(t) > 1]
        toks = list(dict.fromkeys(toks))
        if not toks:
            return "[]"
        partes = [f'"{t[:5]}"*' if len(t) >= 6 else f'"{t}"' for t in toks]
        rows = self.fts.execute("SELECT serie_id, bm25(fts, 0.0, 2.0, 1.0) AS s FROM fts "
                                "WHERE fts MATCH ? ORDER BY s LIMIT ?", (" OR ".join(partes), k)).fetchall()
        return json.dumps([{"serie_id": s, "titulo": str(cat.loc[s, "serie_titulo"]),
                            "distribucion": str(cat.loc[s, "distribucion_titulo"]),
                            "frecuencia": str(cat.loc[s, "frecuencia"])} for s, _ in rows], ensure_ascii=False)

    def _api(self, args):
        params = json.loads(clave(args))
        for intento in range(3):
            try:
                r = requests.get(BASE, params=params, headers=HEADERS, timeout=30)
                if r.status_code in (200, 400):
                    return r.status_code, r.json()
            except Exception:
                pass
            time.sleep(1 + intento)
        return None, None

    def get_series(self, args):
        extra = set(args) - PARAMS_TOOL
        if extra or "ids" not in args:
            return json.dumps({"status": 400, "error": f"Parámetros no permitidos o faltantes: {sorted(extra) or ['ids']}"},
                              ensure_ascii=False)
        k = clave(args)
        fila = self.cache.execute("SELECT status, cuerpo FROM cache WHERE clave=?", (k,)).fetchone()
        if fila:
            status, js = fila[0], json.loads(fila[1])
        else:
            self.fallos_cache += 1
            status, js = self._api(args)
            if status is None:
                self.sin_red += 1
                return json.dumps({"status": None, "error": "no se pudo consultar la API"}, ensure_ascii=False)
            self.cache.execute("INSERT OR REPLACE INTO cache VALUES (?,?,?,?)",
                               (k, status, json.dumps(js, ensure_ascii=False),
                                time.strftime("%Y-%m-%dT%H:%M:%S+00:00", time.gmtime())))
            self.cache.commit()
        if status != 200:
            return json.dumps({"status": status, "error": js.get("errors", js)}, ensure_ascii=False)
        sid = str(args["ids"])
        c = cat.loc[sid] if sid in cat.index else None
        return json.dumps({"status": 200, "serie_id": sid,
                           "titulo": None if c is None else str(c["serie_titulo"]),
                           "unidades": None if (c is None or pd.isna(c["serie_unidades"])) else c["serie_unidades"],
                           "frecuencia": None if c is None else str(c["frecuencia"]),
                           "data": js.get("data", [])}, ensure_ascii=False)

H = Herramientas()

# ---------- Evaluador (el mismo validado con el agente oráculo) ----------
def periodo(f, freq):
    f = "" if f is None else str(f)
    return f[:4] if freq == "anual" else f[:7]

def valor_fecha(tipo, freq, d, params):
    if tipo == "sin_datos" or not d or d[0][1] is None or d[-1][1] is None:
        return None, None
    if tipo == "ultimo_dato":
        return d[0][1], d[0][0]
    if tipo == "valor_puntual":
        return d[0][1], params["start_date"]
    return round(d[-1][1] * 100, 2), d[-1][0]

def cerca(a, b, tol_abs):
    return abs(a - b) <= max(tol_abs, 1e-3 * abs(b))

def puntuar(tipo, freq, g, sub, ids_ultima, objetivo):
    r = {"termino": sub is not None, "serie_ok": ids_ultima == objetivo}
    if sub is None:
        r.update(valor_ok=False, valor_flex=False, fecha_ok=False)
    elif tipo == "sin_datos":
        nulo = sub.get("valor") is None
        r.update(valor_ok=nulo, valor_flex=nulo, fecha_ok=sub.get("fecha") is None)
    else:
        v = sub.get("valor")
        num = isinstance(v, (int, float)) and not isinstance(v, bool)
        tol = 0.011 if tipo == "variacion_interanual" else 5e-5
        ok_e = bool(num and cerca(v, g["valor"], tol))
        ok_f = ok_e or bool(tipo == "variacion_interanual" and num and cerca(v, g["valor"] / 100, 1e-4))
        r.update(valor_ok=ok_e, valor_flex=ok_f,
                 fecha_ok=sub.get("fecha") is not None and periodo(sub["fecha"], freq) == periodo(g["fecha"], freq))
    r["correcta"] = bool(r["termino"] and r["serie_ok"] and r["valor_ok"] and r["fecha_ok"])
    r["correcta_flex"] = bool(r["termino"] and r["serie_ok"] and r["valor_flex"] and r["fecha_ok"])
    return r

def causa(r):
    if r["correcta"]:
        return "correcta"
    if not r["termino"]:
        return "no_termino"
    if not r["serie_ok"]:
        return "serie_equivocada"
    if not r["valor_ok"]:
        return "convencion_porcentaje" if r["valor_flex"] else "valor_equivocado"
    return "fecha_equivocada"

# ---------- Agente ----------
def generar(mensajes):
    ids = tokenizer.apply_chat_template(mensajes, tools=tools, add_generation_prompt=True,
                                        return_tensors="pt").to("cuda")
    n_in = ids.shape[1]
    if n_in > MAX_SEQ_CARGADO - MAX_NUEVOS:
        return None, n_in, 0, 0.0
    t = time.time()
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=MAX_NUEVOS, do_sample=False)
    nuevos = out[0][n_in:]
    return tokenizer.decode(nuevos, skip_special_tokens=False), n_in, len(nuevos), time.time() - t

RE_LLAMADA = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.S)

def episodio(pregunta, sistema):
    msgs = ([{"role": "system", "content": sistema}] if sistema else []) + [{"role": "user", "content": pregunta}]
    log = {"pasos": 0, "n_search": 0, "n_get": 0, "formato_invalido": False, "sin_llamada": False,
           "contexto_excedido": False, "submit": None, "ids_ultima": None, "ultimo_texto": ""}
    for paso in range(1, MAX_PASOS + 1):
        log["pasos"] = paso
        with lf.start_as_current_observation(as_type="generation", name=f"generacion-{paso}",
                                             model=ETIQUETA_MODELO, input=msgs[-1]) as gen:
            txt, n_in, n_out, seg = generar(msgs)
            gen.update(output=txt, metadata={"tokens_entrada": n_in, "tokens_salida": n_out, "segundos": round(seg, 2)})
        if txt is None:
            log["contexto_excedido"] = True
            break
        txt = txt.replace("<|im_end|>", "").strip()
        log["ultimo_texto"] = txt
        m = RE_LLAMADA.search(txt)
        if not m:
            log["sin_llamada"] = True
            break
        try:
            llamada = json.loads(m.group(1))
            nombre, args = llamada["name"], llamada["arguments"]
            assert isinstance(args, dict)
        except Exception:
            log["formato_invalido"] = True
            break
        msgs.append({"role": "assistant", "content": "",
                     "tool_calls": [{"id": f"call_{paso}", "type": "function",
                                     "function": {"name": nombre, "arguments": args}}]})
        if nombre == "submit_answer":
            log["submit"] = args
            break
        with lf.start_as_current_observation(as_type="span", name=f"herramienta:{nombre}", input=args) as ts:
            if nombre == "search_series":
                log["n_search"] += 1
                q = args.get("query")
                res = H.search_series(q) if isinstance(q, str) else json.dumps({"status": 400, "error": "query debe ser texto"})
            elif nombre == "get_series":
                log["n_get"] += 1
                log["ids_ultima"] = str(args.get("ids"))
                res = H.get_series(args)
            else:
                res = json.dumps({"status": 400, "error": f"herramienta desconocida: {nombre}"}, ensure_ascii=False)
            ts.update(output=res[:1500])
        msgs.append({"role": "tool", "tool_call_id": f"call_{paso}", "name": nombre, "content": res})
    return log, msgs

# ---------- Preguntas ----------
Fv = pd.read_csv(f"{RUTA}/eval_{SPLIT}_final2.csv")
Fv = Fv[Fv["marca_var"].fillna("") != "var_sobre_tasa"]
if ESTILO == "coloquial":
    Fv = Fv[~Fv["coloquial_ambigua"]]
S = pd.read_csv(f"{RUTA}/specs_{SPLIT}_v2.csv")
S["idx"] = S.index
Fv["idx"] = Fv["spec_id"].str.split("-").str[1].astype(int)
Fv = Fv.merge(S[["idx", "data"]], on="idx", how="left")
if N_PRUEBA:
    Fv = Fv.sample(frac=1, random_state=4).groupby("tipo").head(int(np.ceil(N_PRUEBA / 4)))
Fv = Fv.reset_index(drop=True)
if PROMPTS == ["basico"] and N_MUESTRA_BASICO:
    Fv = Fv.sample(n=min(N_MUESTRA_BASICO, len(Fv)), random_state=4).reset_index(drop=True)
print(f"Preguntas: {len(Fv)} ({SPLIT}, estilo {ESTILO}) | por tipo: {Fv['tipo'].value_counts().to_dict()}")

def compacto(msgs):
    for m in msgs:
        if m["role"] == "system":
            print("  [sistema] (prompt optimizado)")
        elif m["role"] == "user":
            print("  USUARIO:", m["content"][:150])
        elif m.get("tool_calls"):
            f = m["tool_calls"][0]["function"]
            print("  LLAMADA:", f["name"], json.dumps(f["arguments"], ensure_ascii=False)[:230])
        else:
            print("  RESULTADO:", m["content"][:170])

resultados, trazas = [], []
SCORE_ERR, SCORE_OK = None, 0
for nombre_prompt in PROMPTS:
    t_prompt, fc0 = time.time(), H.fallos_cache
    for r in Fv.to_dict("records"):
        params, d_gold = json.loads(r["params"]), json.loads(r["data"])
        gv, gf = valor_fecha(r["tipo"], r["frecuencia"], d_gold, params)
        g = {"valor": gv, "fecha": gf}
        pregunta = r[ESTILO]
        t0 = time.time()
        with lf.start_as_current_observation(
                as_type="span", name="pregunta", input={"pregunta": pregunta},
                metadata={"modelo": ETIQUETA_MODELO, "prompt": nombre_prompt, "tipo": r["tipo"],
                          "estilo": ESTILO, "split": SPLIT, "spec_id": r["spec_id"]}) as span:
            tid = lf.get_current_trace_id()
            log, msgs = episodio(pregunta, SISTEMA[nombre_prompt])
            sc = puntuar(r["tipo"], r["frecuencia"], g, log["submit"], log["ids_ultima"], params["ids"])
            span.update(output={"submit": log["submit"], "correcta": sc["correcta"], "causa": causa(sc)})
            try:
                span.score_trace(name="correcta", value=float(sc["correcta"]))
                SCORE_OK += 1
            except Exception as e:
                SCORE_ERR = f"{type(e).__name__}: {str(e)[:100]}"
        resultados.append({"prompt": nombre_prompt, "spec_id": r["spec_id"], "tipo": r["tipo"],
                           "frecuencia": r["frecuencia"], **sc, "causa": causa(sc),
                           **{k: log[k] for k in ["pasos", "n_search", "n_get", "formato_invalido",
                                                  "sin_llamada", "contexto_excedido"]},
                           "segundos": round(time.time() - t0, 1), "trace_id": tid})
        trazas.append({"prompt": nombre_prompt, "spec_id": r["spec_id"], "tipo": r["tipo"],
                       "pregunta": pregunta, "messages": msgs, "log": log, "score": sc})
        if len(resultados) % 25 == 0:
            pd.DataFrame(resultados).to_csv(f"{RUTA}/baseline_{SPLIT}_{ESTILO}_parcial.csv", index=False)
            H.cache.commit()
            print(f"  [{len(resultados)}] guardado parcial", flush=True)
    print(f"\n[{nombre_prompt}] terminado en {time.time() - t_prompt:.0f}s | fallos de caché (consultas nuevas a la API): {H.fallos_cache - fc0}")

lf.flush()
R = pd.DataFrame(resultados)

# ---------- Resumen ----------
print("\n" + "=" * 70)
for nombre_prompt in PROMPTS:
    D = R[R["prompt"] == nombre_prompt]
    print(f"\n### Prompt {nombre_prompt} (n = {len(D)})")
    print(f"correcta: {D['correcta'].mean():.0%} | correcta (acepta fracción): {D['correcta_flex'].mean():.0%} | "
          f"terminó con submit_answer: {D['termino'].mean():.0%} | serie correcta: {D['serie_ok'].mean():.0%}")
    print("por tipo:", (D.groupby("tipo")["correcta"].mean() * 100).round(0).to_dict())
    print("causas:", D["causa"].value_counts().to_dict())
    print(f"pasos (prom.): {D['pasos'].mean():.1f} | JSON inválido: {int(D['formato_invalido'].sum())} | "
          f"sin llamada: {int(D['sin_llamada'].sum())} | contexto excedido: {int(D['contexto_excedido'].sum())} | "
          f"segundos por pregunta: {D['segundos'].mean():.1f}")

print("\n" + "=" * 70 + "\nEjemplos de episodios (compactos)")
for nombre_prompt in PROMPTS:
    for tipo in ["variacion_interanual", "sin_datos"]:
        t = next((x for x in trazas if x["prompt"] == nombre_prompt and x["tipo"] == tipo), None)
        if t:
            print(f"\n--- [{nombre_prompt}] {tipo} | causa: {causa(t['score'])}")
            compacto(t["messages"])
            if t["log"]["sin_llamada"]:
                print("  (sin llamada) texto del modelo:", t["log"]["ultimo_texto"][:200])

print("\nScores en Langfuse:", "ok" if SCORE_ERR is None else f"NO funcionó ({SCORE_ERR})")
for t in R["trace_id"].head(2):
    try:
        print("URL de traza:", lf.get_trace_url(trace_id=t))
    except Exception as e:
        print("No pude armar la URL:", type(e).__name__)

# ---------- Guardado ----------
etiqueta = "prueba" if N_PRUEBA else "completa"
R.to_csv(f"{RUTA}/baseline_{SPLIT}_{ESTILO}_{etiqueta}.csv", index=False)
with open(f"{RUTA}/baseline_{SPLIT}_{ESTILO}_{etiqueta}_episodios.jsonl", "w") as f:
    for t in trazas:
        f.write(json.dumps(t, ensure_ascii=False, default=str) + "\n")
if SAVE_CACHE:
    H.cache.commit()
    shutil.copy("/content/cache_api.db", f"{RUTA}/cache_api.db")
    print("Caché actualizada en Drive")
print("Guardado:", f"baseline_{SPLIT}_{ESTILO}_{etiqueta}.csv y _episodios.jsonl")

In [ ]:
print("CELDA: REVISIÓN DE EPISODIOS DEL BASELINE (SIN GPU, SIN COSTO)")
import json
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
S = pd.read_csv(f"{RUTA}/specs_val_v2.csv")
E = [json.loads(l) for l in open(f"{RUTA}/baseline_val_tecnica_prueba_episodios.jsonl")]
R = pd.read_csv(f"{RUTA}/baseline_val_tecnica_prueba.csv")

def gold(spec_id):
    s = S.iloc[int(spec_id.split("-")[1])]
    return json.loads(s["params"])

filas = []
for e in E:
    objetivo = gold(e["spec_id"])["ids"]
    en_cand, ultimo_get = False, None
    for m in e["messages"]:
        if m["role"] == "tool" and m.get("name") == "search_series":
            try:
                if any(c["serie_id"] == objetivo for c in json.loads(m["content"])):
                    en_cand = True
            except Exception:
                pass
        if m.get("tool_calls"):
            f = m["tool_calls"][0]["function"]
            if f["name"] == "get_series":
                ultimo_get = str(f["arguments"].get("ids"))
    filas.append({"prompt": e["prompt"], "spec_id": e["spec_id"], "tipo": e["tipo"],
                  "objetivo_en_candidatos": en_cand, "uso_serie_objetivo": ultimo_get == objetivo})
D = pd.DataFrame(filas).merge(R[["prompt", "spec_id", "causa", "pasos", "n_search", "n_get"]],
                              on=["prompt", "spec_id"])

print("\nDiagnóstico de la serie (por prompt):")
g = D.groupby("prompt").agg(n=("spec_id", "size"),
                            objetivo_en_candidatos=("objetivo_en_candidatos", "mean"),
                            uso_serie_objetivo=("uso_serie_objetivo", "mean"))
print((g.assign(objetivo_en_candidatos=lambda x: (x["objetivo_en_candidatos"] * 100).round(0),
                uso_serie_objetivo=lambda x: (x["uso_serie_objetivo"] * 100).round(0))).to_string())
print("\nCuando la serie SÍ estaba entre los candidatos, ¿la usó?")
print(D[D["objetivo_en_candidatos"]].groupby("prompt")["uso_serie_objetivo"].agg(["size", "mean"]).round(2).to_string())

def mostrar(e, causa):
    print("=" * 70)
    print(f"[{e['prompt']}] {e['tipo']} | causa: {causa}")
    print("PREGUNTA:", e["pregunta"][:200])
    print("GOLD:", json.dumps(gold(e["spec_id"]), ensure_ascii=False)[:220])
    for m in e["messages"]:
        if m.get("tool_calls"):
            f = m["tool_calls"][0]["function"]
            print("  LLAMADA:", f["name"], json.dumps(f["arguments"], ensure_ascii=False)[:260])
        elif m["role"] == "tool":
            print("  RESULTADO:", m["content"][:300])
    if e["log"].get("sin_llamada") and e["log"].get("ultimo_texto"):
        print("  TEXTO FINAL:", e["log"]["ultimo_texto"][:250])

causas = {(r.prompt, r.spec_id): r.causa for r in R.itertuples()}
fallidos = [e for e in E if causas[(e["prompt"], e["spec_id"])] != "correcta"]
prioridad = [e for e in fallidos if e["tipo"] == "ultimo_dato"] + \
            [e for e in fallidos if e["tipo"] != "ultimo_dato"][:4]
print(f"\nEpisodios fallidos: {len(fallidos)} | se muestran {len(prioridad)} (primero los ultimo_dato)")
for e in prioridad:
    mostrar(e, causas[(e["prompt"], e["spec_id"])])

In [ ]:
print("CELDA: DIAGNÓSTICO DEL LÍMITE DE 2048 TOKENS")
assert "model" in globals(), "Ejecutá antes la celda 1"
import torch

print("max_seq_length del modelo:", getattr(model, "max_seq_length", None))
print("max_position_embeddings:", model.config.max_position_embeddings)
print("sliding_window:", getattr(model.config, "sliding_window", None))
print("contexto pedido al cargar:", MAX_SEQ_CARGADO)

relleno = "El índice de precios implícitos de la oferta y demanda globales. "
def probar(n_objetivo):
    texto = relleno * (n_objetivo // 10 + 1)
    ids = tokenizer(texto, return_tensors="pt", add_special_tokens=False)["input_ids"][:, :n_objetivo].to("cuda")
    try:
        with torch.no_grad():
            out = model.generate(ids, max_new_tokens=20, do_sample=False)
        return "ok"
    except Exception as e:
        return f"FALLA {type(e).__name__}: {str(e)[:80]}"

print("\nAntes de tocar nada:")
for n in [1500, 2000, 2040, 2100, 2600, 3500]:
    print(f"  entrada de {n} tokens ->", probar(n))

model.max_seq_length = 4096
print("\nDespués de fijar model.max_seq_length = 4096:")
for n in [2100, 2600, 3500]:
    print(f"  entrada de {n} tokens ->", probar(n))

In [ ]:
print("CELDA: RECARGA DEL MODELO CON 4096 Y COMPROBACIÓN REAL")
assert "FastLanguageModel" in globals(), "Ejecutá antes la celda 1"
import gc, torch

if "model" in globals():
    del model
gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODELO, max_seq_length=4096, load_in_4bit=True)
FastLanguageModel.for_inference(model)
MAX_SEQ_CARGADO = 4096
print("max_seq_length del modelo (real):", getattr(model, "max_seq_length", None))
print("memoria de GPU:", round(torch.cuda.memory_allocated() / 1e9, 1), "GB")

relleno = "El índice de precios implícitos de la oferta y demanda globales. "
def probar(n_objetivo):
    texto = relleno * (n_objetivo // 10 + 1)
    ids = tokenizer(texto, return_tensors="pt", add_special_tokens=False)["input_ids"][:, :n_objetivo].to("cuda")
    try:
        with torch.no_grad():
            model.generate(ids, max_new_tokens=20, do_sample=False)
        return "ok"
    except Exception as e:
        return f"FALLA {type(e).__name__}: {str(e)[:70]}"

for n in [1500, 2040, 2100, 2600, 3500, 3900]:
    print(f"  entrada de {n} tokens ->", probar(n))

In [ ]:
print("max_seq_length real del modelo:", model.max_seq_length)

In [ ]:
print("CELDA 2 de 2: AGENTE + EVALUADOR + CORRIDA CON LANGFUSE (v2)")
assert all(x in globals() for x in ["model", "tokenizer", "lf", "RUTA", "tools"]), "Ejecutá antes la CELDA 1 de 2"
assert model.max_seq_length >= 4096, f"El modelo quedó con max_seq_length={model.max_seq_length}: hay que recargarlo"
import re, json, time, sqlite3, shutil, unicodedata
import logging, warnings
from collections import Counter
import numpy as np
import pandas as pd
import requests
import torch
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

# ---------- Configuración ----------
SPLIT = "val"
ESTILO = "tecnica"            # "tecnica" o "coloquial"
N_PRUEBA = None               # None = todo el split; un número = prueba chica
PROMPTS = ["optimizado"]      # luego, en otra corrida: ["basico"]
N_MUESTRA_BASICO = 100        # solo se usa cuando PROMPTS == ["basico"]
MAX_PASOS = 6
MAX_NUEVOS = 300
SAVE_CACHE = True
ETIQUETA_MODELO = "qwen3-4b-instruct-2507-base"
BASE = "https://apis.datos.gob.ar/series/api/series"
HEADERS = {"User-Agent": "Mozilla/5.0"}
ETIQUETA = ("prueba" if N_PRUEBA else "completa") + "_" + "-".join(PROMPTS)

PROMPT_OPT = """Sos un asistente que responde preguntas sobre estadísticas oficiales de Argentina usando tres herramientas. Seguí siempre este procedimiento:
1. Llamá a search_series con 2 a 6 palabras clave del tema (sin fechas ni números). Elegí, entre los candidatos devueltos, la serie cuyo título, distribución y frecuencia (mensual, trimestral o anual) mejor coincidan con lo que se pide. Usá solo ids devueltos por search_series; nunca inventes ids.
2. Llamá a get_series con la serie elegida:
- Valor de un período: start_date y end_date iguales, en formato AAAA (series anuales) o AAAA-MM.
- Último dato disponible: limit=1 y sort="desc", sin fechas.
- Variación interanual: representation_mode="percent_change_a_year_ago", start_date un año antes del período pedido y end_date el período pedido, con fechas completas AAAA-MM-DD (día 01; en series trimestrales, el primer mes del trimestre). La API devuelve una fracción (0.05 equivale a 5%) y la última fila corresponde al período pedido.
3. Llamá a submit_answer con la respuesta, el valor y la fecha del dato (formato AAAA-MM, o AAAA si la serie es anual). Para variaciones, el valor va en porcentaje (la fracción multiplicada por 100).
Si get_series devuelve data vacía, la API no tiene datos de esa serie para ese período: entregá submit_answer con valor y fecha en null y explicalo en la respuesta. No inventes valores."""
SISTEMA = {"basico": None, "optimizado": PROMPT_OPT}

def norm(s):
    s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s.lower().replace("_", " ")).strip()

STOP = {"de", "del", "la", "el", "los", "las", "en", "y", "a", "por", "que", "cual",
        "fue", "cuanto", "es", "un", "una", "para", "con", "se", "al", "lo"}
PARAMS_TOOL = {"ids", "start_date", "end_date", "representation_mode", "limit", "sort"}
cat = (pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
         .drop_duplicates("serie_id").set_index("serie_id"))

def clave(params):
    p = {k: str(v) for k, v in params.items() if v is not None}
    return json.dumps(dict(sorted(p.items())), ensure_ascii=False)

# ---------- Herramientas: caché primero, API real si falta ----------
class Herramientas:
    def __init__(self):
        self.cache = sqlite3.connect("/content/cache_api.db")
        self.fts = sqlite3.connect("/content/busqueda.db")
        self.fallos_cache, self.sin_red = 0, 0

    def search_series(self, query, k=10):
        toks = [t for t in re.findall(r"[a-z0-9]+", norm(query)) if t not in STOP and len(t) > 1]
        toks = list(dict.fromkeys(toks))
        if not toks:
            return "[]"
        partes = [f'"{t[:5]}"*' if len(t) >= 6 else f'"{t}"' for t in toks]
        rows = self.fts.execute("SELECT serie_id, bm25(fts, 0.0, 2.0, 1.0) AS s FROM fts "
                                "WHERE fts MATCH ? ORDER BY s LIMIT ?", (" OR ".join(partes), k)).fetchall()
        return json.dumps([{"serie_id": s, "titulo": str(cat.loc[s, "serie_titulo"]),
                            "distribucion": str(cat.loc[s, "distribucion_titulo"]),
                            "frecuencia": str(cat.loc[s, "frecuencia"])} for s, _ in rows], ensure_ascii=False)

    def _api(self, args):
        params = json.loads(clave(args))
        for intento in range(3):
            try:
                r = requests.get(BASE, params=params, headers=HEADERS, timeout=30)
                if r.status_code in (200, 400):
                    return r.status_code, r.json()
            except Exception:
                pass
            time.sleep(1 + intento)
        return None, None

    def get_series(self, args):
        extra = set(args) - PARAMS_TOOL
        if extra or "ids" not in args:
            return json.dumps({"status": 400, "error": f"Parámetros no permitidos o faltantes: {sorted(extra) or ['ids']}"},
                              ensure_ascii=False)
        k = clave(args)
        fila = self.cache.execute("SELECT status, cuerpo FROM cache WHERE clave=?", (k,)).fetchone()
        if fila:
            status, js = fila[0], json.loads(fila[1])
        else:
            self.fallos_cache += 1
            status, js = self._api(args)
            if status is None:
                self.sin_red += 1
                return json.dumps({"status": None, "error": "no se pudo consultar la API"}, ensure_ascii=False)
            self.cache.execute("INSERT OR REPLACE INTO cache VALUES (?,?,?,?)",
                               (k, status, json.dumps(js, ensure_ascii=False),
                                time.strftime("%Y-%m-%dT%H:%M:%S+00:00", time.gmtime())))
            self.cache.commit()
        if status != 200:
            return json.dumps({"status": status, "error": js.get("errors", js)}, ensure_ascii=False)
        sid = str(args["ids"])
        c = cat.loc[sid] if sid in cat.index else None
        return json.dumps({"status": 200, "serie_id": sid,
                           "titulo": None if c is None else str(c["serie_titulo"]),
                           "unidades": None if (c is None or pd.isna(c["serie_unidades"])) else c["serie_unidades"],
                           "frecuencia": None if c is None else str(c["frecuencia"]),
                           "data": js.get("data", [])}, ensure_ascii=False)

H = Herramientas()

# ---------- Evaluador (el mismo validado con el agente oráculo) ----------
def periodo(f, freq):
    f = "" if f is None else str(f)
    return f[:4] if freq == "anual" else f[:7]

def valor_fecha(tipo, freq, d, params):
    if tipo == "sin_datos" or not d or d[0][1] is None or d[-1][1] is None:
        return None, None
    if tipo == "ultimo_dato":
        return d[0][1], d[0][0]
    if tipo == "valor_puntual":
        return d[0][1], params["start_date"]
    return round(d[-1][1] * 100, 2), d[-1][0]

def cerca(a, b, tol_abs):
    return abs(a - b) <= max(tol_abs, 1e-3 * abs(b))

def puntuar(tipo, freq, g, sub, ids_ultima, objetivo):
    r = {"termino": sub is not None, "serie_ok": ids_ultima == objetivo}
    if sub is None:
        r.update(valor_ok=False, valor_flex=False, fecha_ok=False)
    elif tipo == "sin_datos":
        nulo = sub.get("valor") is None
        r.update(valor_ok=nulo, valor_flex=nulo, fecha_ok=sub.get("fecha") is None)
    else:
        v = sub.get("valor")
        num = isinstance(v, (int, float)) and not isinstance(v, bool)
        tol = 0.011 if tipo == "variacion_interanual" else 5e-5
        ok_e = bool(num and cerca(v, g["valor"], tol))
        ok_f = ok_e or bool(tipo == "variacion_interanual" and num and cerca(v, g["valor"] / 100, 1e-4))
        r.update(valor_ok=ok_e, valor_flex=ok_f,
                 fecha_ok=sub.get("fecha") is not None and periodo(sub["fecha"], freq) == periodo(g["fecha"], freq))
    r["correcta"] = bool(r["termino"] and r["serie_ok"] and r["valor_ok"] and r["fecha_ok"])
    r["correcta_flex"] = bool(r["termino"] and r["serie_ok"] and r["valor_flex"] and r["fecha_ok"])
    return r

def causa(r):
    if r["correcta"]:
        return "correcta"
    if not r["termino"]:
        return "no_termino"
    if not r["serie_ok"]:
        return "serie_equivocada"
    if not r["valor_ok"]:
        return "convencion_porcentaje" if r["valor_flex"] else "valor_equivocado"
    return "fecha_equivocada"

# ---------- Agente ----------
def generar(mensajes):
    ids = tokenizer.apply_chat_template(mensajes, tools=tools, add_generation_prompt=True,
                                        return_tensors="pt").to("cuda")
    n_in = ids.shape[1]
    if n_in > model.max_seq_length - MAX_NUEVOS:
        return None, n_in, 0, 0.0
    t = time.time()
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=MAX_NUEVOS, do_sample=False)
    nuevos = out[0][n_in:]
    return tokenizer.decode(nuevos, skip_special_tokens=False), n_in, len(nuevos), time.time() - t

RE_LLAMADA = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.S)

def episodio(pregunta, sistema):
    msgs = ([{"role": "system", "content": sistema}] if sistema else []) + [{"role": "user", "content": pregunta}]
    log = {"pasos": 0, "n_search": 0, "n_get": 0, "formato_invalido": False, "sin_llamada": False,
           "contexto_excedido": False, "submit": None, "ids_ultima": None, "ultimo_texto": ""}
    for paso in range(1, MAX_PASOS + 1):
        log["pasos"] = paso
        with lf.start_as_current_observation(as_type="generation", name=f"generacion-{paso}",
                                             model=ETIQUETA_MODELO, input=msgs[-1]) as gen:
            txt, n_in, n_out, seg = generar(msgs)
            gen.update(output=txt, metadata={"tokens_entrada": n_in, "tokens_salida": n_out, "segundos": round(seg, 2)})
        if txt is None:
            log["contexto_excedido"] = True
            break
        txt = txt.replace("<|im_end|>", "").strip()
        log["ultimo_texto"] = txt
        m = RE_LLAMADA.search(txt)
        if not m:
            log["sin_llamada"] = True
            break
        try:
            llamada = json.loads(m.group(1))
            nombre, args = llamada["name"], llamada["arguments"]
            assert isinstance(args, dict)
        except Exception:
            log["formato_invalido"] = True
            break
        msgs.append({"role": "assistant", "content": "",
                     "tool_calls": [{"id": f"call_{paso}", "type": "function",
                                     "function": {"name": nombre, "arguments": args}}]})
        if nombre == "submit_answer":
            log["submit"] = args
            break
        with lf.start_as_current_observation(as_type="span", name=f"herramienta:{nombre}", input=args) as ts:
            if nombre == "search_series":
                log["n_search"] += 1
                q = args.get("query")
                res = H.search_series(q) if isinstance(q, str) else json.dumps({"status": 400, "error": "query debe ser texto"})
            elif nombre == "get_series":
                log["n_get"] += 1
                log["ids_ultima"] = str(args.get("ids"))
                res = H.get_series(args)
            else:
                res = json.dumps({"status": 400, "error": f"herramienta desconocida: {nombre}"}, ensure_ascii=False)
            ts.update(output=res[:1500])
        msgs.append({"role": "tool", "tool_call_id": f"call_{paso}", "name": nombre, "content": res})
    return log, msgs

# ---------- Preguntas ----------
Fv = pd.read_csv(f"{RUTA}/eval_{SPLIT}_final2.csv")
Fv = Fv[Fv["marca_var"].fillna("") != "var_sobre_tasa"]
if ESTILO == "coloquial":
    Fv = Fv[~Fv["coloquial_ambigua"]]
S = pd.read_csv(f"{RUTA}/specs_{SPLIT}_v2.csv")
S["idx"] = S.index
Fv["idx"] = Fv["spec_id"].str.split("-").str[1].astype(int)
Fv = Fv.merge(S[["idx", "data"]], on="idx", how="left")
if N_PRUEBA:
    Fv = Fv.sample(frac=1, random_state=4).groupby("tipo").head(int(np.ceil(N_PRUEBA / 4)))
Fv = Fv.reset_index(drop=True)
if PROMPTS == ["basico"] and N_MUESTRA_BASICO:
    Fv = Fv.sample(n=min(N_MUESTRA_BASICO, len(Fv)), random_state=4).reset_index(drop=True)
print(f"Preguntas: {len(Fv)} ({SPLIT}, estilo {ESTILO}) | por tipo: {Fv['tipo'].value_counts().to_dict()} | prompts: {PROMPTS}")

def compacto(msgs):
    for m in msgs:
        if m["role"] == "system":
            print("  [sistema] (prompt optimizado)")
        elif m["role"] == "user":
            print("  USUARIO:", m["content"][:150])
        elif m.get("tool_calls"):
            f = m["tool_calls"][0]["function"]
            print("  LLAMADA:", f["name"], json.dumps(f["arguments"], ensure_ascii=False)[:230])
        else:
            print("  RESULTADO:", m["content"][:170])

resultados, trazas = [], []
SCORE_ERR, SCORE_OK = None, 0
t_total = time.time()
for nombre_prompt in PROMPTS:
    t_prompt, fc0 = time.time(), H.fallos_cache
    for r in Fv.to_dict("records"):
        params, d_gold = json.loads(r["params"]), json.loads(r["data"])
        gv, gf = valor_fecha(r["tipo"], r["frecuencia"], d_gold, params)
        g = {"valor": gv, "fecha": gf}
        pregunta = r[ESTILO]
        t0 = time.time()
        with lf.start_as_current_observation(
                as_type="span", name="pregunta", input={"pregunta": pregunta},
                metadata={"modelo": ETIQUETA_MODELO, "prompt": nombre_prompt, "tipo": r["tipo"],
                          "estilo": ESTILO, "split": SPLIT, "spec_id": r["spec_id"]}) as span:
            tid = lf.get_current_trace_id()
            log, msgs = episodio(pregunta, SISTEMA[nombre_prompt])
            sc = puntuar(r["tipo"], r["frecuencia"], g, log["submit"], log["ids_ultima"], params["ids"])
            span.update(output={"submit": log["submit"], "correcta": sc["correcta"], "causa": causa(sc)})
            try:
                span.score_trace(name="correcta", value=float(sc["correcta"]))
                SCORE_OK += 1
            except Exception as e:
                SCORE_ERR = f"{type(e).__name__}: {str(e)[:100]}"
        resultados.append({"prompt": nombre_prompt, "spec_id": r["spec_id"], "tipo": r["tipo"],
                           "frecuencia": r["frecuencia"], **sc, "causa": causa(sc),
                           **{k: log[k] for k in ["pasos", "n_search", "n_get", "formato_invalido",
                                                  "sin_llamada", "contexto_excedido"]},
                           "segundos": round(time.time() - t0, 1), "trace_id": tid})
        trazas.append({"prompt": nombre_prompt, "spec_id": r["spec_id"], "tipo": r["tipo"],
                       "pregunta": pregunta, "messages": msgs, "log": log, "score": sc})
        if len(resultados) % 25 == 0:
            pd.DataFrame(resultados).to_csv(f"{RUTA}/baseline_{SPLIT}_{ESTILO}_{ETIQUETA}_parcial.csv", index=False)
            H.cache.commit()
            print(f"  [{len(resultados)}/{len(Fv) * len(PROMPTS)}] guardado parcial | {time.time() - t_total:.0f}s", flush=True)
    print(f"\n[{nombre_prompt}] terminado en {time.time() - t_prompt:.0f}s | fallos de caché (consultas nuevas a la API): {H.fallos_cache - fc0}")

lf.flush()
R = pd.DataFrame(resultados)

# ---------- Resumen ----------
print("\n" + "=" * 70)
for nombre_prompt in PROMPTS:
    D = R[R["prompt"] == nombre_prompt]
    print(f"\n### Prompt {nombre_prompt} (n = {len(D)})")
    print(f"correcta: {D['correcta'].mean():.0%} | correcta (acepta fracción): {D['correcta_flex'].mean():.0%} | "
          f"terminó con submit_answer: {D['termino'].mean():.0%} | serie correcta: {D['serie_ok'].mean():.0%}")
    print("por tipo:", (D.groupby("tipo")["correcta"].mean() * 100).round(0).to_dict())
    print("causas:", D["causa"].value_counts().to_dict())
    print(f"pasos (prom.): {D['pasos'].mean():.1f} | JSON inválido: {int(D['formato_invalido'].sum())} | "
          f"sin llamada: {int(D['sin_llamada'].sum())} | contexto excedido: {int(D['contexto_excedido'].sum())} | "
          f"segundos por pregunta: {D['segundos'].mean():.1f}")

print("\n" + "=" * 70 + "\nEjemplos de episodios (compactos)")
for nombre_prompt in PROMPTS:
    for tipo in ["variacion_interanual", "sin_datos"]:
        t = next((x for x in trazas if x["prompt"] == nombre_prompt and x["tipo"] == tipo), None)
        if t:
            print(f"\n--- [{nombre_prompt}] {tipo} | causa: {causa(t['score'])}")
            compacto(t["messages"])
            if t["log"]["sin_llamada"]:
                print("  (sin llamada) texto del modelo:", t["log"]["ultimo_texto"][:200])

print("\nScores en Langfuse:", "ok" if SCORE_ERR is None else f"NO funcionó ({SCORE_ERR})")
for t in R["trace_id"].head(2):
    try:
        print("URL de traza:", lf.get_trace_url(trace_id=t))
    except Exception as e:
        print("No pude armar la URL:", type(e).__name__)

# ---------- Guardado ----------
R.to_csv(f"{RUTA}/baseline_{SPLIT}_{ESTILO}_{ETIQUETA}.csv", index=False)
with open(f"{RUTA}/baseline_{SPLIT}_{ESTILO}_{ETIQUETA}_episodios.jsonl", "w") as f:
    for t in trazas:
        f.write(json.dumps(t, ensure_ascii=False, default=str) + "\n")
if SAVE_CACHE:
    H.cache.commit()
    shutil.copy("/content/cache_api.db", f"{RUTA}/cache_api.db")
    print("Caché actualizada en Drive")
print("Guardado:", f"baseline_{SPLIT}_{ESTILO}_{ETIQUETA}.csv y _episodios.jsonl")
print(f"Tiempo total: {(time.time() - t_total) / 60:.0f} min")

In [ ]:
print("CELDA: ANÁLISIS DEL BASELINE (SIN GPU, SIN COSTO)")
import json
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
ARCH = "baseline_val_tecnica_completa_optimizado"
R = pd.read_csv(f"{RUTA}/{ARCH}.csv")
E = {e["spec_id"]: e for e in (json.loads(l) for l in open(f"{RUTA}/{ARCH}_episodios.jsonl"))}
S = pd.read_csv(f"{RUTA}/specs_val_v2.csv")

def igual(a, b):
    return a is not None and b is not None and abs(a - b) <= 1e-9 * max(1, abs(b))

def ultimo_get(msgs):
    args, res = None, None
    for i, m in enumerate(msgs):
        if m.get("tool_calls") and m["tool_calls"][0]["function"]["name"] == "get_series":
            args, res = m["tool_calls"][0]["function"]["arguments"], None
            if i + 1 < len(msgs) and msgs[i + 1]["role"] == "tool":
                try:
                    res = json.loads(msgs[i + 1]["content"])
                except Exception:
                    res = None
    return args, res

def recupero(tipo, d_gold, args, res, objetivo):
    if args is None or str(args.get("ids")) != objetivo or not res or res.get("status") != 200:
        return False
    d = res.get("data", [])
    if tipo == "sin_datos":
        return d == []
    if not d or not d_gold:
        return False
    i = 0 if tipo in ("ultimo_dato", "valor_puntual") else -1
    return d[i][0] == d_gold[i][0] and igual(d[i][1], d_gold[i][1])

filas = []
for r in R.to_dict("records"):
    s = S.iloc[int(r["spec_id"].split("-")[1])]
    params, d_gold = json.loads(s["params"]), json.loads(s["data"])
    e = E[r["spec_id"]]
    args, res = ultimo_get(e["messages"])
    rec = recupero(r["tipo"], d_gold, args, res, params["ids"])
    grupo = ("A_correcta" if r["correcta"] else
             "B_dato_recuperado_no_entregado" if rec and not r["termino"] else
             "C_dato_recuperado_entregado_mal" if rec else "D_dato_no_recuperado")
    txt = (e["log"].get("ultimo_texto") or "").strip()
    filas.append({**r, "grupo": grupo, "recupero": rec,
                  "texto_final": ("json_suelto" if txt.startswith("{") else "prosa") if r["sin_llamada"] else "-",
                  "args_ultima": json.dumps(args, ensure_ascii=False) if args else "", "msgs": e["messages"]})
D = pd.DataFrame(filas)

print(f"Episodios: {len(D)} | correctas (evaluador): {int(D['correcta'].sum())}")
print("\nGrupos:", D["grupo"].value_counts().sort_index().to_dict())
print("\nGrupo por tipo:")
print(pd.crosstab(D["tipo"], D["grupo"]).to_string())
print("\nCausa del evaluador por tipo:")
print(pd.crosstab(D["tipo"], D["causa"]).to_string())
print(f"\nRecuperó el dato correcto (con o sin entregarlo): {D['recupero'].mean():.0%} "
      f"| por tipo: {(D.groupby('tipo')['recupero'].mean() * 100).round(0).to_dict()}")
print("\nCuando no llamó a submit_answer, el texto final era:", D[D["sin_llamada"]]["texto_final"].value_counts().to_dict())

print("\nEpisodios con contexto excedido:")
for _, r in D[D["contexto_excedido"]].head(5).iterrows():
    ult = r["msgs"][-1]
    print(f"  [{r['tipo']}] última consulta: {r['args_ultima'][:140]} | respuesta de {len(ult.get('content', ''))} caracteres")

print("\nDato recuperado pero entregado mal (hasta 6):")
for _, r in D[D["grupo"] == "C_dato_recuperado_entregado_mal"].head(6).iterrows():
    sub = json.loads(json.dumps(r["msgs"][-1]["tool_calls"][0]["function"]["arguments"], default=str)) if r["msgs"][-1].get("tool_calls") else None
    print(f"  [{r['tipo']}] causa: {r['causa']} | entregó: valor={sub.get('valor') if sub else None} fecha={sub.get('fecha') if sub else None}")